# EmoWave - SVM Pipeline (Local)

**Tai hien bai bao Wang et al. (2014):**
Power Spectrum + Asymmetry Features + LDS Smoothing + SVM

---

**Yeu cau:** File `s01.dat` -> `s32.dat` dat trong thu muc `data/deap/`

## 1. Cau hinh

In [ ]:
import os

# --- CAU HINH ---
PROCESSED_DIR = "../data/processed"       # Duong dan toi thu muc chua file .npy
USE_LDS = True                   # LDS smoothing
USE_GRIDSEARCH = True            # GridSearch (tat de chay nhanh)
SEARCH_RBF = False               # False: Chi GridSearch Linear (nhanh, khuyen dung). True: GridSearch ca RBF (RAT CHAM)
MODE = "subject-dependent"      # "subject-dependent": Train model rieng cho tung subject (khuyen dung, giong bai bao gốc)
                                 # "subject-independent": Train tren 31 subjects, test tren 1 subject (LOSO)
                                 # "subject-mixed": Tron du lieu tat ca subjects vao train chung
RESULTS_DIR = "../results"

# Kiem tra data
files = ["X_epochs.npy", "y_valence.npy", "y_arousal.npy", "subject_groups.npy"]
print(f"Kiem tra cac file trong {PROCESSED_DIR}:")
for f in files:
    path = os.path.join(PROCESSED_DIR, f)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f"  [OK] {f} ({size_mb:.1f} MB)")
    else:
        print(f"  [MISSING] {f}")

## 2. Import Libraries

In [ ]:
import numpy as np
import pickle
import json
import time
import gc
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC, LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
)
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score
)

# Khoi tao bien toan cuc de luu trong so model
global_weights = {}

print("All imports OK")

## 3. DEAP Loader

In [ ]:
SFREQ = 128

def load_processed_data():
    print(f"Loading data from {PROCESSED_DIR}...")
    X_epochs = np.load(os.path.join(PROCESSED_DIR, "X_epochs.npy"))
    y_valence = np.load(os.path.join(PROCESSED_DIR, "y_valence.npy"))
    y_arousal = np.load(os.path.join(PROCESSED_DIR, "y_arousal.npy"))
    subject_groups = np.load(os.path.join(PROCESSED_DIR, "subject_groups.npy"))
    
    print(f"Loaded X_epochs: {X_epochs.shape}")
    return X_epochs, y_valence, y_arousal, subject_groups

print("DEAP Loader ready")

## 4. Feature Extraction

In [ ]:
FREQ_BANDS = {
    'delta': (0.5, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta':  (13, 30),
    'gamma': (30, 45),
}

ASYMMETRY_PAIRS = [
    (0, 16), (1, 17), (2, 18), (3, 19), (4, 20), (5, 21), (6, 22),
    (7, 23), (8, 24), (9, 25), (10, 26), (11, 27), (12, 28), (13, 29),
]


def extract_power_spectrum(epoch, sfreq=SFREQ):
    fft_vals = np.abs(np.fft.rfft(epoch, axis=-1)) ** 2
    freqs = np.fft.rfftfreq(epoch.shape[-1], 1.0 / sfreq)
    band_powers = []
    for (f_low, f_high) in FREQ_BANDS.values():
        idx = np.where((freqs >= f_low) & (freqs <= f_high))[0]
        if len(idx) == 0:
            band_powers.append(np.zeros(epoch.shape[0]))
        else:
            band_powers.append(np.log1p(fft_vals[:, idx].mean(axis=-1)))
    return np.stack(band_powers, axis=-1)


def extract_asymmetry(power_features):
    asymmetry = []
    for left_idx, right_idx in ASYMMETRY_PAIRS:
        asymmetry.append(power_features[left_idx] - power_features[right_idx])
    return np.array(asymmetry)


def extract_all_features(epoch):
    power = extract_power_spectrum(epoch)
    asym = extract_asymmetry(power)
    return np.concatenate([power.flatten(), asym.flatten()])


def extract_features_dataset(X_epochs):
    n = len(X_epochs)
    print(f"  Extracting {n} epochs...")
    start = time.time()
    features = []
    for i in range(n):
        features.append(extract_all_features(X_epochs[i]))
        if (i + 1) % 10000 == 0:
            elapsed = time.time() - start
            speed = (i+1) / elapsed
            eta = (n - i - 1) / speed
            print(f"    {i+1}/{n}  ({speed:.0f} eps/s, ETA: {eta:.0f}s)")
    result = np.array(features, dtype=np.float32)
    print(f"  Done! Shape: {result.shape} ({time.time()-start:.1f}s)")
    return result

print("Feature extractors ready")

## 5. LDS Smoothing

In [ ]:
def lds_smoothing(features_seq, alpha=0.3):
    smoothed = np.zeros_like(features_seq)
    smoothed[0] = features_seq[0]
    for t in range(1, len(features_seq)):
        smoothed[t] = alpha * features_seq[t] + (1 - alpha) * smoothed[t - 1]
    return smoothed


def apply_lds(X_features, epochs_per_trial=60, alpha=0.3):
    X_smoothed = np.copy(X_features)
    n_trials = len(X_features) // epochs_per_trial
    for t in range(n_trials):
        s, e = t * epochs_per_trial, (t + 1) * epochs_per_trial
        X_smoothed[s:e] = lds_smoothing(X_features[s:e], alpha)
    return X_smoothed

print("LDS ready")

## 6. SVM Training

In [ ]:
def train_svm_simple(X_train, y_train, kernel="linear", C=1.0):
    """Train SVM don gian. Dung LinearSVC cho linear kernel (nhanh gap 50x)."""
    if kernel == "linear":
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('svm', LinearSVC(C=C, class_weight='balanced', max_iter=5000, random_state=42, dual='auto'))
        ])
    else:
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('svm', SVC(kernel=kernel, C=C, class_weight='balanced', random_state=42))
        ])
    pipeline.fit(X_train, y_train)
    return pipeline


def train_svm_gridsearch(X_train, y_train):
    """GridSearch: Linear (full data) + RBF (subsample neu qua lon)."""
    n_train = len(X_train)

    # --- Linear ---
    print("  GridSearch Linear...")
    linear_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', LinearSVC(class_weight='balanced', max_iter=5000, random_state=42, dual='auto'))
    ])
    linear_grid = GridSearchCV(
        linear_pipe, {'svm__C': [0.01, 0.1, 1, 10, 100]},
        cv=StratifiedKFold(5, shuffle=True, random_state=42),
        scoring='accuracy', n_jobs=-1, verbose=0, refit=True
    )
    t0 = time.time()
    linear_grid.fit(X_train, y_train)
    print(f"    Best C={linear_grid.best_params_['svm__C']}"
          f" -> {linear_grid.best_score_*100:.2f}% ({time.time()-t0:.1f}s)")

    if not SEARCH_RBF:
        return linear_grid.best_estimator_, \
               {'kernel': 'linear', **linear_grid.best_params_}, \
               linear_grid.best_score_

    # --- RBF (subsample neu data lon) ---
    print("  GridSearch RBF...")
    MAX_RBF = 15000
    if n_train > MAX_RBF:
        print(f"    Subsample {MAX_RBF}/{n_train} (tranh qua tai)...")
        idx = np.random.RandomState(42).choice(n_train, MAX_RBF, replace=False)
        X_sub, y_sub = X_train[idx], y_train[idx]
    else:
        X_sub, y_sub = X_train, y_train

    rbf_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel='rbf', class_weight='balanced', random_state=42))
    ])
    rbf_grid = GridSearchCV(
        rbf_pipe,
        {'svm__C': [1, 10, 100], 'svm__gamma': ['scale', 0.01]},
        cv=StratifiedKFold(3, shuffle=True, random_state=42),
        scoring='accuracy', n_jobs=-1, verbose=0, refit=True
    )
    t0 = time.time()
    rbf_grid.fit(X_sub, y_sub)
    print(f"    Best C={rbf_grid.best_params_['svm__C']:.2f},"
          f" gamma={rbf_grid.best_params_['svm__gamma']}"
          f" -> {rbf_grid.best_score_*100:.2f}% ({time.time()-t0:.1f}s)")

    # --- Chon tot nhat ---
    if linear_grid.best_score_ >= rbf_grid.best_score_:
        print(f"  >> Chon: Linear ({linear_grid.best_score_*100:.2f}%)")
        return linear_grid.best_estimator_, \
               {'kernel': 'linear', **linear_grid.best_params_}, \
               linear_grid.best_score_
    else:
        print(f"  >> Chon: RBF ({rbf_grid.best_score_*100:.2f}%)")
        if n_train > MAX_RBF:
            print(f"    [Luu y] De tranh treo may, dung luon model RBF train tren {MAX_RBF} samples.")
            best = rbf_grid.best_estimator_
        else:
            best = rbf_grid.best_estimator_
        return best, {'kernel': 'rbf', **rbf_grid.best_params_}, rbf_grid.best_score_


## 7. Evaluation

In [ ]:
def evaluate_and_plot(y_test, y_pred, label_type="2class",
                      best_params=None, cv_score=None):
    os.makedirs(RESULTS_DIR, exist_ok=True)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')

    names = ["Negative", "Positive"] if label_type == "2class" \
            else ["Vui ve", "So hai", "Buon", "Thu gian"]

    print(f"\n  {'='*40}")
    print(f"  SVM --- {label_type} ({MODE} mode)")
    print(f"  {'='*40}")
    print(f"  Accuracy:      {acc*100:.2f}%")
    print(f"  F1 (weighted): {f1*100:.2f}%")
    if best_params:
        if isinstance(best_params, list):
            print(f"  Best params:   (Average / Multiple models across subjects)")
        else:
            print(f"  Best params:   {best_params}")
    print(classification_report(y_test, y_pred, labels=[0,1] if label_type=="2class" else [0,1,2,3], target_names=names, zero_division=0))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred, labels=[0,1] if label_type=="2class" else [0,1,2,3])
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=names, yticklabels=names, ax=ax)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Actual', fontsize=12)
    ax.set_title(f'SVM Confusion Matrix ({label_type} - {MODE})\n'
                 f'Accuracy: {acc*100:.2f}%', fontsize=14)
    plt.tight_layout()

    fig_path = os.path.join(RESULTS_DIR, f"confusion_matrix_svm_{label_type}.png")
    plt.savefig(fig_path, dpi=150)
    plt.show()

    # JSON
    results = {
        "model": "SVM", "label_type": label_type, "mode": MODE,
        "accuracy": round(acc, 4), "f1_weighted": round(f1, 4),
        "best_params": str(best_params) if best_params else None,
        "cv_score": round(cv_score, 4) if cv_score else None,
        "confusion_matrix": cm.tolist(),
        "classification_report": classification_report(
            y_test, y_pred, labels=[0,1] if label_type=="2class" else [0,1,2,3], target_names=names, output_dict=True, zero_division=0),
    }
    json_path = os.path.join(RESULTS_DIR, f"svm_results_{label_type}.json")
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"  Saved: {fig_path}")
    print(f"  Saved: {json_path}")
    return results

print("Evaluation ready")

## 8. Unified Pipeline Execution Function

In [ ]:
def get_labels_from_val_aro(y_val, y_aro, label_type):
    if label_type == "2class":
        return y_val
    else:
        # 4-class logic mapping:
        # val=1, aro=1 -> 0
        # val=0, aro=1 -> 1
        # val=0, aro=0 -> 2
        # val=1, aro=0 -> 3
        classes = np.zeros(len(y_val), dtype=int)
        classes[(y_val == 1) & (y_aro == 1)] = 0
        classes[(y_val == 0) & (y_aro == 1)] = 1
        classes[(y_val == 0) & (y_aro == 0)] = 2
        classes[(y_val == 1) & (y_aro == 0)] = 3
        return classes

def run_pipeline(label_type="2class"):
    print("=" * 60)
    print(f"  EmoWave SVM Pipeline --- {label_type} ({MODE} mode)")
    print("=" * 60)
    
    # 1. Load Data
    print("\n[1] Loading Preprocessed Data...")
    X_epochs, y_val, y_aro, groups = load_processed_data()
    y_all = get_labels_from_val_aro(y_val, y_aro, label_type)
    
    if MODE == "subject-mixed":
        print("\n[2] Feature extraction (All)...")
        X_feat = extract_features_dataset(X_epochs)
        del X_epochs; gc.collect()
        
        if USE_LDS:
            print("\n[3] LDS smoothing...")
            X_feat = apply_lds(X_feat)
            
        print("\n[4] Train/test split...")
        X_train, X_test, y_train, y_test = train_test_split(
            X_feat, y_all, test_size=0.2, random_state=42, stratify=y_all)
        del X_feat; gc.collect()
        
        print("\n[5] Training SVM...")
        if USE_GRIDSEARCH:
            model, params, cv_score = train_svm_gridsearch(X_train, y_train)
        else:
            model = train_svm_simple(X_train, y_train)
            params, cv_score = None, None
            
        try:
            svm_step = model.named_steps['svm']
            if hasattr(svm_step, 'coef_'):
                global_weights[label_type] = svm_step.coef_[0]
        except:
            pass
            
        print("\n[6] Evaluation...")
        y_pred = model.predict(X_test)
        results = evaluate_and_plot(y_test, y_pred, label_type, params, cv_score)
        return results
        
    elif MODE == "subject-independent":
        # Nhom cuoi cung (31) lam test, con lai lam train
        test_group = 31
        print(f"\n[2] LOSO Mode: Train tren 31 subjects (0..30), Test tren subject {test_group}")
        
        train_idx = (groups != test_group)
        test_idx = (groups == test_group)
        
        X_train_ep, y_train = X_epochs[train_idx], y_all[train_idx]
        X_test_ep, y_test = X_epochs[test_idx], y_all[test_idx]
        del X_epochs, y_all; gc.collect()
        
        print(f"  Train shape: X={X_train_ep.shape}, y={y_train.shape}")
        print(f"  Test shape: X={X_test_ep.shape}, y={y_test.shape}")
        
        print("\n[3] Feature extraction (Train)...")
        X_train_feat = extract_features_dataset(X_train_ep)
        del X_train_ep; gc.collect()
        
        print("\n[3] Feature extraction (Test)...")
        X_test_feat = extract_features_dataset(X_test_ep)
        del X_test_ep; gc.collect()
        
        if USE_LDS:
            print("\n[4] LDS smoothing...")
            X_train_feat = apply_lds(X_train_feat)
            X_test_feat = apply_lds(X_test_feat)
            
        print("\n[5] Training SVM...")
        if USE_GRIDSEARCH:
            model, params, cv_score = train_svm_gridsearch(X_train_feat, y_train)
        else:
            model = train_svm_simple(X_train_feat, y_train)
            params, cv_score = None, None
            
        print("\n[6] Evaluation...")
        scaler = model.named_steps['scaler']
        svm = model.named_steps['svm']
        X_test_scaled = scaler.transform(X_test_feat)
        y_pred = svm.predict(X_test_scaled)
        
        results = evaluate_and_plot(y_test, y_pred, label_type, params, cv_score)
        return results
        
    else:
        # subject-dependent
        unique_groups = np.unique(groups)
        print(f"\n[2] Huan luyen models rieng biet cho {len(unique_groups)} subjects...")
        t_start = time.time()
        
        y_test_all, y_pred_all, best_params_all, subject_weights = [], [], [], []
        
        for sg in unique_groups:
            idx = (groups == sg)
            X_sub_ep = X_epochs[idx]
            y_sub = y_all[idx]
            
            if len(np.unique(y_sub)) < 2:
                print(f"  Subject {sg:02d}: Bo qua vi chi co 1 nhan")
                continue
                
            X_sub_feat = extract_features_dataset(X_sub_ep)
            if USE_LDS:
                X_sub_feat = apply_lds(X_sub_feat)
                
            X_train, X_test, y_train, y_test = train_test_split(
                X_sub_feat, y_sub, test_size=0.2, random_state=42, stratify=y_sub)
            
            if USE_GRIDSEARCH:
                model, params, _ = train_svm_gridsearch(X_train, y_train)
                best_params_all.append(params)
            else:
                model = train_svm_simple(X_train, y_train)
                
            try:
                svm_step = model.named_steps['svm']
                if hasattr(svm_step, 'coef_'):
                    subject_weights.append(svm_step.coef_[0])
            except:
                pass
                
            scaler = model.named_steps['scaler']
            svm = model.named_steps['svm']
            X_test_scaled = scaler.transform(X_test)
            y_pred = svm.predict(X_test_scaled)
            
            y_test_all.append(y_test)
            y_pred_all.append(y_pred)
            acc = accuracy_score(y_test, y_pred)
            print(f"  Subject {sg:02d}: Acc = {acc*100:.2f}%")
            
        y_test_combined = np.concatenate(y_test_all, axis=0)
        y_pred_combined = np.concatenate(y_pred_all, axis=0)
        print(f"\n  >> Da xong {len(unique_groups)} subjects! Tong thoi gian: {time.time()-t_start:.1f}s")
        
        if len(subject_weights) > 0:
            global_weights[label_type] = np.mean(subject_weights, axis=0)
            
        results = evaluate_and_plot(y_test_combined, y_pred_combined, label_type, best_params_all)
        return results

print("run_pipeline function ready")


---
## 9. CHAY PIPELINE - 2 CLASS (Positive / Negative)

In [ ]:
results_2c = run_pipeline(label_type="2class")

## 10. CHAY PIPELINE - 4 CLASS (Vui / So / Buon / Thu gian)

In [ ]:
results_4c = run_pipeline(label_type="4class")

## 11. Tong ket

In [ ]:
print("=" * 60)
print(f"  SVM Pipeline DONE! Mode: {MODE}")
print("=" * 60)
print(f"\n  {'Label':<10} {'Accuracy':>10} {'F1':>10}")
print(f"  {'---':<10} {'---':>10} {'---':>10}")
print(f"  {'2class':<10} {results_2c['accuracy']*100:>9.2f}% {results_2c['f1_weighted']*100:>9.2f}%")
print(f"  {'4class':<10} {results_4c['accuracy']*100:>9.2f}% {results_4c['f1_weighted']*100:>9.2f}%")
print(f"\n  Results saved in: {os.path.abspath(RESULTS_DIR)}/")

## 12. Feature Importance Analysis (Do quan trong cua vung nao)

In [ ]:
if "2class" in global_weights:
    bands = ['delta', 'theta', 'alpha', 'beta', 'gamma']
    channels = [
        'Fp1', 'AF3', 'F3', 'F7', 'FC5', 'FC1', 'C3', 'T7', 'CP5', 'CP1', 'P3', 'P7', 'PO3', 'O1', 'Oz', 'Pz',
        'Fp2', 'AF4', 'F4', 'F8', 'FC6', 'FC2', 'C4', 'T8', 'CP6', 'CP2', 'P4', 'P8', 'PO4', 'O2', 'FCz', 'Cz'
    ]
    asym_pairs = [
        ('Fp1', 'Fp2'), ('AF3', 'AF4'), ('F3', 'F4'), ('F7', 'F8'), 
        ('FC5', 'FC6'), ('FC1', 'FC2'), ('C3', 'C4'), ('T7', 'T8'), 
        ('CP5', 'CP6'), ('CP1', 'CP2'), ('P3', 'P4'), ('P7', 'P8'), 
        ('PO3', 'PO4'), ('O1', 'O2')
    ]

    feature_names = []
    # 160 PSD features
    for ch in channels:
        for b in bands:
            feature_names.append(f"PSD_{ch}_{b}")
    # 70 Asymmetry features
    for left, right in asym_pairs:
        for b in bands:
            feature_names.append(f"ASYM_{left}-{right}_{b}")

    # Lay trong so
    svm_coef = global_weights["2class"]
    sorted_idx = np.argsort(svm_coef)

    print("=" * 60)
    print("  DO QUAN TRONG CUA CAC DAC TRUNG EEG (2-CLASS)")
    print("=" * 60)
    print("\n[+] Top 10 dac trung ung ho cam xuc tich cuc (Positive) cao nhat:")
    for idx in sorted_idx[-10:][::-1]:
        print(f"  {feature_names[idx]:<25}: Trong so = {svm_coef[idx]:.4f}")

    print("\n[-] Top 10 dac trung ung ho cam xuc tieu cuc (Negative) cao nhat:")
    for idx in sorted_idx[:10]:
        print(f"  {feature_names[idx]:<25}: Trong so = {svm_coef[idx]:.4f}")
else:
    print("Vui long chay xong Pipeline 2-class de trich xuat trong so.")